# Lab 5 - Kodowanie Huffmana

Implementacja kodowania Huffmana na podstawie szablonu z Lab 4.

In [1]:
import math
import pickle
import heapq
from bitarray import bitarray
from collections import Counter

class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        # Dla kolejki priorytetowej
        return self.freq < other.freq

class HuffmanCompressor:
    def __init__(self):
        self.char_to_code = {}
        self.code_to_char = {}
        self.root = None

    def create(self, frequencies):
        """Tworzy drzewo Huffmana i kody na podstawie częstości występowania znaków."""
        if not frequencies:
            return

        # Tworzymy kolejkę priorytetową z liści
        priority_queue = [HuffmanNode(char, freq) for char, freq in frequencies.items()]
        heapq.heapify(priority_queue)

        # Budujemy drzewo
        while len(priority_queue) > 1:
            left = heapq.heappop(priority_queue)
            right = heapq.heappop(priority_queue)

            merged = HuffmanNode(None, left.freq + right.freq)
            merged.left = left
            merged.right = right

            heapq.heappush(priority_queue, merged)

        self.root = priority_queue[0]
        self.char_to_code = {}
        self._build_codes(self.root, "")
        self.code_to_char = {v: k for k, v in self.char_to_code.items()}
        
        print(f"Liczba unikalnych znaków: {len(frequencies)}")

    def _build_codes(self, node, current_code):
        if node is None:
            return

        # Jeśli to liść, przypisujemy kod
        if node.char is not None:
            # Specjalny przypadek: tylko jeden unikalny znak w tekście
            self.char_to_code[node.char] = current_code if current_code else "0"
            return

        self._build_codes(node.left, current_code + "0")
        self._build_codes(node.right, current_code + "1")

    def encode(self, text):
        """Zamienia tekst na bitarray."""
        bit_string = "".join(self.char_to_code[char] for char in text)
        return bitarray(bit_string)

    def decode(self, bits, original_text_len):
        """Dekoduje bity z powrotem na tekst."""
        if not self.root:
            return ""
            
        decoded_chars = []
        current_node = self.root
        bits_str = bits.to01()
        bit_idx = 0
        
        # Dekodujemy aż do uzyskania original_text_len znaków
        while len(decoded_chars) < original_text_len and bit_idx < len(bits_str):
            bit = bits_str[bit_idx]
            bit_idx += 1
            
            if bit == '0':
                current_node = current_node.left
            else:
                current_node = current_node.right
            
            if current_node.char is not None:
                decoded_chars.append(current_node.char)
                current_node = self.root
                
        return "".join(decoded_chars)

    def save(self, filename, bits, original_len):
        """Zapisuje kod i dane do pliku."""
        data = {
            'char_to_code': self.char_to_code,
            'encoded_bits': bits.tobytes(),
            'num_bits': len(bits),
            'original_len': original_len
        }
        with open(filename, 'wb') as f:
            pickle.dump(data, f)

    def load(self, filename):
        """Wczytuje kod i dane z pliku."""
        with open(filename, 'rb') as f:
            data = pickle.load(f)
        
        self.char_to_code = data['char_to_code']
        self.code_to_char = {v: k for k, v in self.char_to_code.items()}
        
        # Odtwarzamy drzewo na podstawie kodów
        self._rebuild_tree()
        
        bits = bitarray()
        bits.frombytes(data['encoded_bits'])
        # Przycinamy do faktycznej liczby bitów (padding bajtowy)
        bits = bits[:data['num_bits']]
        return bits, data['original_len']

    def _rebuild_tree(self):
        """Odtwarza drzewo Huffmana z mapowania char_to_code."""
        self.root = HuffmanNode(None, 0)
        for char, code in self.char_to_code.items():
            current_node = self.root
            for bit in code:
                if bit == '0':
                    if current_node.left is None:
                        current_node.left = HuffmanNode(None, 0)
                    current_node = current_node.left
                else:
                    if current_node.right is None:
                        current_node.right = HuffmanNode(None, 0)
                    current_node = current_node.right
            current_node.char = char

def load_text(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return f.read()

def calculate_stats(text, compressor):
    n = len(text)
    freqs = Counter(text)
    probs = {c: f / n for c, f in freqs.items()}
    
    # Entropia H(X)
    entropy = -sum(p * math.log2(p) for p in probs.values())
    
    # Średnia długość kodu L
    avg_len = sum(probs[char] * len(compressor.char_to_code[char]) for char in freqs)
    
    # Efektywność eta
    efficiency = (entropy / avg_len) * 100 if avg_len > 0 else 0
    
    return entropy, avg_len, efficiency

if __name__ == "__main__":
    filename = "norm_wiki_sample.txt"
    original_text = load_text(filename)
    
    # 1. Analiza częstości
    freqs = Counter(original_text)
    
    compressor = HuffmanCompressor()
    compressor.create(freqs)
    
    # 2. Kodowanie
    encoded_data = compressor.encode(original_text)
    
    # 3. Zapis i odczyt
    compressor.save("huffman_compressed.bin", encoded_data, len(original_text))
    
    # Nowy obiekt kompresora do testu odczytu
    new_compressor = HuffmanCompressor()
    loaded_bits, loaded_len = new_compressor.load("huffman_compressed.bin")
    
    # 4. Dekodowanie
    decoded_text = new_compressor.decode(loaded_bits, loaded_len)
    
    # 5. Weryfikacja i statystyki
    print(f"Oryginał (fragment): {original_text[:100]}...")
    print(f"Zdekodowany (fragment): {decoded_text[:100]}...")
    print(f"Czy identyczne? {'TAK' if original_text == decoded_text else 'NIE'}")
    
    entropy, avg_len, efficiency = calculate_stats(original_text, compressor)
    
    print(f"\n--- Statystyki ---")
    print(f"Entropia: {entropy:.4f} bitów/znak")
    print(f"Średnia długość kodu: {avg_len:.4f} bitów/znak")
    print(f"Efektywność: {efficiency:.2f}%")
    
    orig_size_bits = len(original_text) * 8
    comp_size_bits = len(encoded_data)
    print(f"Rozmiar oryginału (8-bit ASCII): {orig_size_bits} bitów")
    print(f"Rozmiar zakodowany: {comp_size_bits} bitów")
    print(f"Stopień kompresji: {orig_size_bits / comp_size_bits:.2f}:1")

Liczba unikalnych znaków: 37
Oryginał (fragment):  albert of prussia 17 may 1490 20 march 1568 was the last grand master of the teutonic knights who a...
Zdekodowany (fragment):  albert of prussia 17 may 1490 20 march 1568 was the last grand master of the teutonic knights who a...
Czy identyczne? TAK

--- Statystyki ---
Entropia: 4.2804 bitów/znak
Średnia długość kodu: 4.3090 bitów/znak
Efektywność: 99.34%
Rozmiar oryginału (8-bit ASCII): 86311528 bitów
Rozmiar zakodowany: 46489714 bitów
Stopień kompresji: 1.86:1


### Teoria i wyniki

**Średnia długość kodu ($L$):**
$$L = \sum_{i=1}^{n} p_i l_i$$
gdzie $p_i$ to prawdopodobieństwo wystąpienia symbolu, a $l_i$ to długość jego kodu w bitach.

**Entropia ($H$):**
$$H = -\sum_{i=1}^{n} p_i \log_2 p_i$$

**Efektywność ($\eta$):**
$$\eta = \frac{H}{L}$$

Kodowanie Huffmana jest kodowaniem optymalnym dla symboli kodowanych pojedynczo, co oznacza, że $H \le L < H + 1$.